## STEP: 4 Cart2Insights — Data Quality Analysis

In [ ]:
## Objective
The objective is to systematically assess the quality of the raw e-commerce datasets before performing data cleaning and transformation.

## To investigate:

>>Missing values
>>Exact duplicates
>>Primary-key uniqueness
>>Foreign-key validity
>>Incorrect Data types
>>Categorical consistency
>>Numerical validity
>>Date validity
>>Potential outliers


## Load the Data

In [4]:
import pandas as pd

customers = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_customers_dataset.csv")
orders = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_orders_dataset.csv")
order_items = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_order_items_dataset.csv")
order_payments = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_order_reviews_dataset.csv")
products = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_products_dataset.csv")
sellers = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_sellers_dataset.csv")
geolocation = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_geolocation_dataset.csv")
category_translation = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\product_category_name_translation.csv")

dataset_shapes = {
    "customers": customers.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "order_payments": order_payments.shape,
    "order_reviews": order_reviews.shape,
    "products": products.shape,
    "sellers": sellers.shape,
    "geolocation": geolocation.shape,
    "translation": category_translation.shape,
}

print(dataset_shapes)

dataset_shapes = {
    "customers": customers.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "order_payments": order_payments.shape,
    "order_reviews": order_reviews.shape,
    "products": products.shape,
    "sellers": sellers.shape,
    "geolocation": geolocation.shape,
    "translation": category_translation.shape,
}

print(dataset_shapes)



{'customers': (99441, 5), 'orders': (99441, 8), 'order_items': (112650, 7), 'order_payments': (103886, 5), 'order_reviews': (99224, 7), 'products': (32951, 9), 'sellers': (3095, 4), 'geolocation': (1000163, 5), 'translation': (71, 2)}
{'customers': (99441, 5), 'orders': (99441, 8), 'order_items': (112650, 7), 'order_payments': (103886, 5), 'order_reviews': (99224, 7), 'products': (32951, 9), 'sellers': (3095, 4), 'geolocation': (1000163, 5), 'translation': (71, 2)}


## 1.IDENTIFY MISSING VALUES

In [5]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(f"MISSING VALUES - {name.upper()}")
    print("=" * 60)

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values")
    else:
        print(missing)


MISSING VALUES - CUSTOMERS
No missing values

MISSING VALUES - ORDERS
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

MISSING VALUES - ORDER_ITEMS
No missing values

MISSING VALUES - ORDER_PAYMENTS
No missing values

MISSING VALUES - ORDER_REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

MISSING VALUES - PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

MISSING VALUES - SELLERS
No missing values

MISSING VALUES - GEOLOCATION
No missing values

MISSING VALUES - CATEGORY_TRANSLATION
No missing values


## MISSING VALUE SUMMARY

In [6]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()
        missing_percentage = (missing_count / len(df)) * 100

        missing_summary.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(missing_percentage, 2)
        })

missing_df = pd.DataFrame(missing_summary)

missing_df = missing_df[missing_df["Missing_Count"] > 0]

print(missing_df)

          Dataset                         Column  Missing_Count  \
9          orders              order_approved_at            160   
10         orders   order_delivered_carrier_date           1783   
11         orders  order_delivered_customer_date           2965   
28  order_reviews           review_comment_title          87656   
29  order_reviews         review_comment_message          58247   
33       products          product_category_name            610   
34       products            product_name_lenght            610   
35       products     product_description_lenght            610   
36       products             product_photos_qty            610   
37       products               product_weight_g              2   
38       products              product_length_cm              2   
39       products              product_height_cm              2   
40       products               product_width_cm              2   

    Missing_Percentage  
9                 0.16  
10         

In [7]:
## Display Table
missing_df.sort_values(
    by=["Dataset", "Missing_Count"],
    ascending=[True, False]
)

,Dataset,Column,Missing_Count,Missing_Percentage
28,order_reviews,review_comment_title,87656,88.34
29,order_reviews,review_comment_message,58247,58.70
11,orders,order_delivered_customer_date,2965,2.98
10,orders,order_delivered_carrier_date,1783,1.79
9,orders,order_approved_at,160,0.16
33,products,product_category_name,610,1.85
34,products,product_name_lenght,610,1.85
35,products,product_description_lenght,610,1.85
36,products,product_photos_qty,610,1.85
37,products,product_weight_g,2,0.01


## 2.CHECK DUPLICATES

In [8]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    print("=" * 60)
    print(f"{name.upper()}")
    print(f"Total rows     : {len(df)}")
    print(f"Duplicate rows : {duplicate_count}")
    print("=" * 60)

CUSTOMERS
Total rows     : 99441
Duplicate rows : 0
ORDERS
Total rows     : 99441
Duplicate rows : 0
ORDER_ITEMS
Total rows     : 112650
Duplicate rows : 0
ORDER_PAYMENTS
Total rows     : 103886
Duplicate rows : 0
ORDER_REVIEWS
Total rows     : 99224
Duplicate rows : 0
PRODUCTS
Total rows     : 32951
Duplicate rows : 0
SELLERS
Total rows     : 3095
Duplicate rows : 0
GEOLOCATION
Total rows     : 1000163
Duplicate rows : 261836
CATEGORY_TRANSLATION
Total rows     : 71
Duplicate rows : 0


## Summary Table

In [9]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Duplicate_Rows": df.duplicated().sum(),
        "Duplicate_Percentage": round(
            (df.duplicated().sum() / len(df)) * 100, 2
        )
    })

duplicate_df = pd.DataFrame(duplicate_summary)

duplicate_df

,Dataset,Total_Rows,Duplicate_Rows,Duplicate_Percentage
0,customers,99441,0,0.00
1,orders,99441,0,0.00
2,order_items,112650,0,0.00
3,order_payments,103886,0,0.00
4,order_reviews,99224,0,0.00
5,products,32951,0,0.00
6,sellers,3095,0,0.00
7,geolocation,1000163,261836,26.18
8,category_translation,71,0,0.00


# 3. CHECK INCORRECT DATA TYPES

In [10]:

for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(f"DATA TYPES - {name.upper()}")
    print("=" * 70)
    print(df.dtypes)


DATA TYPES - CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

DATA TYPES - ORDERS
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

DATA TYPES - ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

DATA TYPES - ORDER_PAYMENTS
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object



#  DATA TYPE SUMMARY

In [11]:
dtype_summary = []

for name, df in datasets.items():
    for column in df.columns:
        dtype_summary.append({
            "Dataset": name,
            "Column": column,
            "Data_Type": str(df[column].dtype)
        })

dtype_df = pd.DataFrame(dtype_summary)

dtype_df

,Dataset,Column,Data_Type
0,customers,customer_id,str
1,customers,customer_unique_id,str
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,str
4,customers,customer_state,str
5,orders,order_id,str
6,orders,customer_id,str
7,orders,order_status,str
8,orders,order_purchase_timestamp,str
9,orders,order_approved_at,str


In [ ]:
| Dataset        | Column                          | Expected type |
| -------------- | ------------------------------- | ------------- |
| orders         | `order_purchase_timestamp`      | datetime      |
| orders         | `order_approved_at`             | datetime      |
| orders         | `order_delivered_carrier_date`  | datetime      |
| orders         | `order_delivered_customer_date` | datetime      |
| orders         | `order_estimated_delivery_date` | datetime      |
| order_items    | `shipping_limit_date`           | datetime      |
| order_items    | `price`                         | numeric       |
| order_items    | `freight_value`                 | numeric       |
| order_payments | `payment_value`                 | numeric       |
| order_payments | `payment_installments`          | integer       |
| products       | `product_weight_g`              | numeric       |
| products       | `product_length_cm`             | numeric       |
| products       | `product_height_cm`             | numeric       |
| products       | `product_width_cm`              | numeric       |
| geolocation    | `geolocation_lat`               | numeric       |
| geolocation    | `geolocation_lng`               | numeric       |


## 4.CHECK INCONSISTENT / INVALID CATEGORICAL VALUES

In [12]:
print("ORDER STATUS VALUES")
print(orders["order_status"].value_counts(dropna=False))

print("\nPAYMENT TYPES")
print(order_payments["payment_type"].value_counts(dropna=False))

print("\nREVIEW SCORES")
print(order_reviews["review_score"].value_counts(dropna=False))

print("\nPRODUCT CATEGORIES")
print(products["product_category_name"].value_counts(dropna=False).head(20))

ORDER STATUS VALUES
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

PAYMENT TYPES
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

REVIEW SCORES
review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

PRODUCT CATEGORIES
product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
p

# CHECK INVALID NUMERIC VALUES

In [13]:
print("ORDER ITEMS - PRICE <= 0")
print(order_items[order_items["price"] <= 0].shape)

print("\nORDER ITEMS - FREIGHT < 0")
print(order_items[order_items["freight_value"] < 0].shape)

print("\nPAYMENTS - PAYMENT VALUE <= 0")
print(order_payments[order_payments["payment_value"] <= 0].shape)

print("\nPAYMENTS - INSTALLMENTS <= 0")
print(order_payments[order_payments["payment_installments"] <= 0].shape)

print("\nREVIEWS OUTSIDE 1-5")
print(
    order_reviews[
        ~order_reviews["review_score"].between(1, 5)
    ].shape
)

ORDER ITEMS - PRICE <= 0
(0, 7)

ORDER ITEMS - FREIGHT < 0
(0, 7)

PAYMENTS - PAYMENT VALUE <= 0
(9, 5)

PAYMENTS - INSTALLMENTS <= 0
(2, 5)

REVIEWS OUTSIDE 1-5
(0, 7)


# CHECK PRODUCT DIMENSIONS

In [14]:

product_numeric = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in product_numeric:
    print(f"\n{col}")
    print("Negative values:", (products[col] < 0).sum())
    print("Zero values    :", (products[col] == 0).sum())


product_weight_g
Negative values: 0
Zero values    : 4

product_length_cm
Negative values: 0
Zero values    : 0

product_height_cm
Negative values: 0
Zero values    : 0

product_width_cm
Negative values: 0
Zero values    : 0


# CHECK GEOLOCATION VALUES

In [15]:


invalid_lat = geolocation[
    ~geolocation["geolocation_lat"].between(-90, 90)
]

invalid_lng = geolocation[
    ~geolocation["geolocation_lng"].between(-180, 180)
]

print("Invalid latitude :", len(invalid_lat))
print("Invalid longitude:", len(invalid_lng))

Invalid latitude : 0
Invalid longitude: 0


# CHECK ORDER DATE CONSISTENCY

In [16]:

invalid_approved = orders[
    orders["order_approved_at"].notna() &
    (orders["order_approved_at"] < orders["order_purchase_timestamp"])
]

invalid_carrier = orders[
    orders["order_delivered_carrier_date"].notna() &
    (orders["order_delivered_carrier_date"] <
     orders["order_approved_at"])
]

invalid_delivery = orders[
    orders["order_delivered_customer_date"].notna() &
    (orders["order_delivered_customer_date"] <
     orders["order_delivered_carrier_date"])
]

print("Approval before purchase:", len(invalid_approved))
print("Carrier before approval:", len(invalid_carrier))
print("Delivery before carrier:", len(invalid_delivery))

Approval before purchase: 1677
Carrier before approval: 8964
Delivery before carrier: 25789


In [17]:
print(
    order_payments["payment_installments"]
    .value_counts()
    .sort_index()
)

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64


## 5. IDENTIFY POTENTIAL OUTLIERS USING IQR

In [18]:

outlier_columns = {
    "order_items": [
        "price",
        "freight_value"
    ],

    "order_payments": [
        "payment_value",
        "payment_installments"
    ],

    "products": [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],

    "geolocation": [
        "geolocation_lat",
        "geolocation_lng"
    ]
}

outlier_summary = []

for dataset_name, columns in outlier_columns.items():

    df = datasets[dataset_name]

    for col in columns:

        data = df[col].dropna()

        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)

        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        outliers = data[
            (data < lower_limit) |
            (data > upper_limit)
        ]

        outlier_summary.append({
            "Dataset": dataset_name,
            "Column": col,
            "Q1": round(Q1, 2),
            "Q3": round(Q3, 2),
            "IQR": round(IQR, 2),
            "Lower_Limit": round(lower_limit, 2),
            "Upper_Limit": round(upper_limit, 2),
            "Outlier_Count": len(outliers),
            "Outlier_Percentage": round(
                len(outliers) / len(data) * 100, 2
            )
        })

outlier_df = pd.DataFrame(outlier_summary)

outlier_df

,Dataset,Column,Q1,Q3,IQR,Lower_Limit,Upper_Limit,Outlier_Count,Outlier_Percentage
0,order_items,price,39.90,134.90,95.00,-102.60,277.40,8427,7.48
1,order_items,freight_value,13.08,21.15,8.07,0.98,33.25,12134,10.77
2,order_payments,payment_value,56.79,171.84,115.05,-115.78,344.41,7981,7.68
3,order_payments,payment_installments,1.00,4.00,3.00,-3.50,8.50,6313,6.08
4,products,product_weight_g,300.00,1900.00,1600.00,-2100.00,4300.00,4551,13.81
5,products,product_length_cm,18.00,38.00,20.00,-12.00,68.00,1380,4.19
6,products,product_height_cm,8.00,21.00,13.00,-11.50,40.50,1892,5.74
7,products,product_width_cm,15.00,30.00,15.00,-7.50,52.50,912,2.77
8,geolocation,geolocation_lat,-23.60,-19.98,3.62,-29.04,-14.54,168240,16.82
9,geolocation,geolocation_lng,-48.57,-43.77,4.81,-55.78,-36.56,42348,4.23


In [19]:
outlier_df[
    outlier_df["Outlier_Count"] > 0
]

,Dataset,Column,Q1,Q3,IQR,Lower_Limit,Upper_Limit,Outlier_Count,Outlier_Percentage
0,order_items,price,39.90,134.90,95.00,-102.60,277.40,8427,7.48
1,order_items,freight_value,13.08,21.15,8.07,0.98,33.25,12134,10.77
2,order_payments,payment_value,56.79,171.84,115.05,-115.78,344.41,7981,7.68
3,order_payments,payment_installments,1.00,4.00,3.00,-3.50,8.50,6313,6.08
4,products,product_weight_g,300.00,1900.00,1600.00,-2100.00,4300.00,4551,13.81
5,products,product_length_cm,18.00,38.00,20.00,-12.00,68.00,1380,4.19
6,products,product_height_cm,8.00,21.00,13.00,-11.50,40.50,1892,5.74
7,products,product_width_cm,15.00,30.00,15.00,-7.50,52.50,912,2.77
8,geolocation,geolocation_lat,-23.60,-19.98,3.62,-29.04,-14.54,168240,16.82
9,geolocation,geolocation_lng,-48.57,-43.77,4.81,-55.78,-36.56,42348,4.23


In [20]:
price_Q1 = order_items["price"].quantile(0.25)
price_Q3 = order_items["price"].quantile(0.75)

price_IQR = price_Q3 - price_Q1

price_upper = price_Q3 + 1.5 * price_IQR

high_price = order_items[
    order_items["price"] > price_upper
]

print("Potential high-price outliers:")
print(high_price[["order_id", "product_id", "price"]]
      .sort_values("price", ascending=False)
      .head(20))

Potential high-price outliers:
                                order_id                        product_id  \
3556    0812eb902a67711a1cb742b3cdaa65ae  489ae2aa008f021502940f251d4cce7f   
112233  fefacc66af859508bf1a7934eab1e97f  69c590f7ffc7bf8db97190b6cb6ed62e   
107841  f5136e38d1a14a4dbd87dff67da82701  1bdf5e6731585cf01aa8169c7028d6ad   
74336   a96610ab360d42a2e5335a3998b4718a  a6492cc69376c469ab6f61d8f44de961   
11249   199af31afc78c699f0dbf71fb178d4d4  c3ed642d592594bb648ff4a04cee2747   
62086   8dbc85d1447242f3b127dda390d56e19  259037a6a41845e455183f89c5035f18   
29193   426a9742b533fc6fed17d1fd6d143d7e  a1beef8f3992dbd4cd8726796aa69c53   
45843   68101694e5c5dc7330c91e1bbc36214f  6cdf8fc1d741c76586d8b6b15e9eef30   
78310   b239ca7cd485940b31882363b52e6674  dd113cb02b2af9c8e5787e8f1f0722f6   
59137   86c4eab1571921a6a6e248ed312f5a5a  6902c1962dd19d540807d0ab8fade5c6   
56663   80dfedb6d17bf23539beeef3c768f4d7  4ca7b91a31637bd24fb8e559d5e015e4   
67456   9a3966c23190dbdbaabed08e8

In [21]:
payment_Q1 = order_payments["payment_value"].quantile(0.25)
payment_Q3 = order_payments["payment_value"].quantile(0.75)

payment_IQR = payment_Q3 - payment_Q1

payment_upper = payment_Q3 + 1.5 * payment_IQR

high_payments = order_payments[
    order_payments["payment_value"] > payment_upper
]

print(high_payments[
    ["order_id", "payment_type", "payment_value"]
].sort_values(
    "payment_value",
    ascending=False
).head(20))

                               order_id payment_type  payment_value
52107  03caa2c082116e1d31e67e9ae3700499  credit_card       13664.08
34370  736e1922ae60d0d6a89247b851902527       boleto        7274.88
41419  0812eb902a67711a1cb742b3cdaa65ae  credit_card        6929.31
49581  fefacc66af859508bf1a7934eab1e97f       boleto        6922.21
85539  f5136e38d1a14a4dbd87dff67da82701       boleto        6726.66
62409  2cc9089445046817a7539d90805e6e5a       boleto        6081.54
43232  a96610ab360d42a2e5335a3998b4718a  credit_card        4950.34
70320  b4c4b76c642808cbe472a32b86cddc95  credit_card        4809.44
6440   199af31afc78c699f0dbf71fb178d4d4  credit_card        4764.34
67546  8dbc85d1447242f3b127dda390d56e19  credit_card        4681.78
69126  426a9742b533fc6fed17d1fd6d143d7e  credit_card        4513.32
91368  d2f270487125ddc41fd134c4003ad1d7   debit_card        4445.50
83794  68101694e5c5dc7330c91e1bbc36214f  credit_card        4175.26
94128  b239ca7cd485940b31882363b52e6674   debit_

## 6.CHECK PRIMARY-KEY UNIQUENESS

In [22]:

primary_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ],
    "category_translation": ["product_category_name"]
}

for dataset_name, keys in primary_keys.items():

    df = datasets[dataset_name]

    print("\n" + "=" * 70)
    print(dataset_name.upper())
    print("=" * 70)

    for key in keys:

        if key in df.columns:

            total = len(df)
            unique = df[key].nunique()
            duplicates = df[key].duplicated().sum()

            print(f"\nColumn: {key}")
            print(f"Total rows       : {total}")
            print(f"Unique values    : {unique}")
            print(f"Duplicate values : {duplicates}")


CUSTOMERS

Column: customer_id
Total rows       : 99441
Unique values    : 99441
Duplicate values : 0

ORDERS

Column: order_id
Total rows       : 99441
Unique values    : 99441
Duplicate values : 0

ORDER_ITEMS

Column: order_id
Total rows       : 112650
Unique values    : 98666
Duplicate values : 13984

Column: order_item_id
Total rows       : 112650
Unique values    : 21
Duplicate values : 112629

ORDER_PAYMENTS

Column: order_id
Total rows       : 103886
Unique values    : 99440
Duplicate values : 4446

Column: payment_sequential
Total rows       : 103886
Unique values    : 29
Duplicate values : 103857

ORDER_REVIEWS

Column: review_id
Total rows       : 99224
Unique values    : 98410
Duplicate values : 814

PRODUCTS

Column: product_id
Total rows       : 32951
Unique values    : 32951
Duplicate values : 0

SELLERS

Column: seller_id
Total rows       : 3095
Unique values    : 3095
Duplicate values : 0

GEOLOCATION

Column: geolocation_zip_code_prefix
Total rows       : 1000163
Uni

# CHECK COMPOSITE PRIMARY KEY - ORDER ITEMS

In [23]:

duplicate_order_items = order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print("Duplicate (order_id, order_item_id):", duplicate_order_items)

Duplicate (order_id, order_item_id): 0


In [24]:
print(
    "Duplicate review_id:",
    order_reviews["review_id"].duplicated().sum()
)

print(
    "Duplicate (review_id, order_id):",
    order_reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

Duplicate review_id: 814
Duplicate (review_id, order_id): 0


In [25]:
print(
    "Duplicate customer_id:",
    customers["customer_id"].duplicated().sum()
)

print(
    "Duplicate customer_unique_id:",
    customers["customer_unique_id"].duplicated().sum()
)

Duplicate customer_id: 0
Duplicate customer_unique_id: 3345
